# Asyncio
Asynchronous I/O, or asyncio, is a library to write concurrent code using the **async/await** syntax. It is the modern standard for handling I/O-bound tasks in Python (e.g., network requests, database queries, file operations).

Unlike Threading, which uses the OS to switch context, asyncio uses cooperative multitasking within a single thread. This makes it extremely efficient for managing thousands of concurrent connections.

## Coroutines
A coroutine is a special function that can pause its execution and yield control back to the event loop. You define it with `async def` and call it using `await`.

In [1]:
import asyncio

async def say_hello():
    print("Hello...")
    await asyncio.sleep(1) # Simulated I/O
    print("...World!")

# In a script, you would run it with:
# asyncio.run(say_hello())

# In Jupyter, the event loop is already running, so we can just await it:
await say_hello()

Hello...
...World!


## Running Tasks Concurrently
Using `await` one by one is still sequential. To run multiple coroutines concurrently, use `asyncio.gather()`.

In [2]:
import time

async def fetch_data(id, delay):
    print(f"Task {id} starting...")
    await asyncio.sleep(delay)
    print(f"Task {id} finished after {delay}s")
    return f"Data {id}"

async def main():
    start = time.perf_counter()
    
    # Schedule multiple calls concurrently
    results = await asyncio.gather(
        fetch_data(1, 2),
        fetch_data(2, 1),
        fetch_data(3, 1.5)
    )
    
    end = time.perf_counter()
    print(f"Results: {results}")
    print(f"Total time: {end - start:.2f}s")

await main()

Task 1 starting...
Task 2 starting...
Task 3 starting...
Task 2 finished after 1s
Task 3 finished after 1.5s
Task 1 finished after 2s
Results: ['Data 1', 'Data 2', 'Data 3']
Total time: 2.01s


## Creating Tasks
`asyncio.create_task()` schedules a coroutine to run on the event loop immediately, without waiting for it to finish. This is useful for fire-and-forget logic or background processing.

In [3]:
async def background_task():
    while True:
        print("Heartbeat...")
        await asyncio.sleep(2)

async def main():
    # This task will start running in the background
    task = asyncio.create_task(background_task())
    
    print("Main logic running...")
    await asyncio.sleep(5)
    print("Main logic done.")
    
    # Cancel the background task before finishing
    task.cancel()

await main()

Main logic running...
Heartbeat...
Heartbeat...
Heartbeat...
Main logic done.


## Handling Timeouts
In production, network requests should always have a timeout to prevent hanging the system.

In [4]:
async def slow_request():
    await asyncio.sleep(10)
    return "Success"

async def main():
    try:
        # Use wait_for to set a limit
        result = await asyncio.wait_for(slow_request(), timeout=2.0)
        print(result)
    except asyncio.TimeoutError:
        print("Request timed out!")

await main()

Request timed out!


## Summary
- `async def`: Defines a coroutine.
- `await`: Pauses execution until the coroutine is done.
- `asyncio.gather()`: Runs multiple coroutines concurrently.
- `asyncio.create_task()`: Backgrounds a task.
- **Pro-tip**: Use `asyncio` for I/O (Web APIs, DBs) and `multiprocessing` for CPU-heavy computations (Math, Image processing).